# AI_CLASSIFY — Order Priority Classification

`AI_CLASSIFY` assigns a label from a provided list to each input text row.

| Item | Detail |
|---|---|
| **Function** | `SNOWFLAKE.CORTEX.AI_CLASSIFY` |
| **Data** | `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS` (zero-ingestion) |
| **Use-case** | Classify free-text order comments into business categories |


## Step 1 — Environment Setup

In [ ]:
CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — Create a View on TPC-H Orders

In [ ]:
CREATE OR REPLACE VIEW orders_sample AS
SELECT
    o_orderkey      AS order_key,
    o_orderstatus   AS order_status,
    o_totalprice    AS total_price,
    o_orderdate     AS order_date,
    o_orderpriority AS order_priority,
    o_comment       AS order_comment
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
LIMIT 500;

In [ ]:
SELECT * FROM orders_sample LIMIT 5;

## Step 3 — Classify Order Comments

Use `AI_CLASSIFY` to categorize each order comment into one of the given labels.

In [ ]:
SELECT
    order_key,
    order_comment,
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        order_comment,
        ['shipping_issue', 'pricing_concern', 'general_inquiry', 'urgent_request', 'positive_feedback']
    ) AS classification
FROM orders_sample
LIMIT 20;

## Step 4 — Aggregate Classification Results

In [ ]:
WITH classified AS (
    SELECT
        order_key,
        order_priority,
        SNOWFLAKE.CORTEX.AI_CLASSIFY(
            order_comment,
            ['shipping_issue', 'pricing_concern', 'general_inquiry', 'urgent_request', 'positive_feedback']
        ):label::STRING AS category
    FROM orders_sample
)
SELECT category, COUNT(*) AS cnt
FROM classified
GROUP BY category
ORDER BY cnt DESC;

## Step 5 — Cross-Reference with Order Priority

In [ ]:
WITH classified AS (
    SELECT
        order_priority,
        SNOWFLAKE.CORTEX.AI_CLASSIFY(
            order_comment,
            ['shipping_issue', 'pricing_concern', 'general_inquiry', 'urgent_request', 'positive_feedback']
        ):label::STRING AS category
    FROM orders_sample
)
SELECT order_priority, category, COUNT(*) AS cnt
FROM classified
GROUP BY order_priority, category
ORDER BY order_priority, cnt DESC;

## Key Takeaways

- `AI_CLASSIFY` returns a JSON object with `label` and `score` fields.
- The label list can be any business-relevant categories — no training needed.
- Combine with GROUP BY for instant analytics dashboards.
- Works on any text column — support tickets, survey responses, log messages.
